# M2 — ResNet-50 category baseline and fine-tuning (Fashion-MNIST, M1 split unchanged)
Run the cells **in order**. Runtime → Change runtime type → **T4 GPU** first.
Results are only what these cells print and save; nothing here is filled in by hand.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to T4 GPU (Runtime > Change runtime type), then run again."
!pip -q install scikit-learn pandas pyyaml pillow

**Step 0.** Mount Google Drive now, so the checkpoints can be saved there (Colab deletes everything on the VM when the session ends). A pop-up asks you to allow access.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/wardiq_m2

**Step 1.** Upload `WARDIQ_M2_Baseline_Handoff.zip` (the M2 package).

In [ ]:
from google.colab import files
up = files.upload()
zipname = [n for n in up if n.endswith(".zip")][0]
!rm -rf /content/wardiq && mkdir -p /content/wardiq && unzip -q -o "{zipname}" -d /content/wardiq
%cd /content/wardiq
!ls

**Step 2.** Upload the M1 files this code needs. You can upload `files.zip` (the four files zipped) or the four files one by one: `taxonomy.json`, `apply_taxonomy.py`, `split_manifest.csv`, and `raw_manifest.csv` (or `clean_manifest.csv`).

In [ ]:
import os, re, shutil, zipfile
os.makedirs("configs/m1", exist_ok=True); os.makedirs("data/manifests", exist_ok=True); os.makedirs("scripts", exist_ok=True)
up = files.upload()
dest = {"taxonomy.json": "configs/m1/taxonomy.json", "apply_taxonomy.py": "scripts/apply_taxonomy.py",
        "split_manifest.csv": "data/manifests/split_manifest.csv",
        "clean_manifest.csv": "data/manifests/clean_manifest.csv", "raw_manifest.csv": "data/manifests/raw_manifest.csv"}
for n in list(up):
    if n.endswith(".zip"):
        zipfile.ZipFile(n).extractall("m1_inputs")
        for root, _, names in os.walk("m1_inputs"):
            for m in names:
                clean = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", m)
                if clean in dest: shutil.copy(os.path.join(root, m), dest[clean])
    else:
        clean = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", n)
        if clean in dest: shutil.move(n, dest[clean])
MANIFEST = "data/manifests/clean_manifest.csv" if os.path.exists("data/manifests/clean_manifest.csv") else "data/manifests/raw_manifest.csv"
for need in ("configs/m1/taxonomy.json", "scripts/apply_taxonomy.py", "data/manifests/split_manifest.csv", MANIFEST):
    print(("OK      " if os.path.exists(need) else "MISSING ") + need)
print("manifest used:", MANIFEST)

**Step 3.** Build the training table (uses the M1 split exactly as given).

In [ ]:
!python scripts/build_classification_manifest.py --manifest {MANIFEST} --datasets "Fashion-MNIST" --label-space source_label --out data/manifests/classification_manifest_fmnist.csv

**Step 4.** Baseline (frozen backbone, head only).

In [ ]:
!python -m src.wardiq.clothing.resnet_category --config configs/m2/resnet50_baseline_fmnist.yaml

**Step 4b.** Save the baseline checkpoint to Drive right away and print its checksum.

In [ ]:
!cp artifacts/m2/resnet50_baseline_fmnist/checkpoint.pt /content/drive/MyDrive/wardiq_m2/baseline_fmnist.pt
!sha256sum /content/drive/MyDrive/wardiq_m2/baseline_fmnist.pt

**Step 5.** Fine-tune (`layer4` only), starting from the unchanged baseline.

In [ ]:
!python -m src.wardiq.clothing.resnet_category --config configs/m2/resnet50_finetune_fmnist.yaml

**Step 5b.** Save the fine-tune checkpoint to Drive and print its checksum.

In [ ]:
!cp artifacts/m2/resnet50_finetune_fmnist/checkpoint.pt /content/drive/MyDrive/wardiq_m2/finetune_fmnist.pt
!sha256sum /content/drive/MyDrive/wardiq_m2/finetune_fmnist.pt

**Step 6.** Print the table rows for `docs/m2/resnet_category.md` (from the saved metrics) and check the reload test.

In [ ]:
import json
for label, run in [("A: Baseline (Fashion-MNIST)", "resnet50_baseline_fmnist"), ("A: Fine-tune layer4 (Fashion-MNIST)", "resnet50_finetune_fmnist")]:
    try:
        m = json.load(open(f"artifacts/m2/{run}/val_metrics.json")); c = m["cost"]
        r = json.load(open(f"artifacts/m2/{run}/reload_check.json"))
        print(f"| {label} | {m['n_items']} | {m['accuracy']:.3f} | {m['macro_f1']:.3f} | {m['best_epoch']} | {c['seconds']}s | {c['device']} |   reload check: {r['matching']}/{r['checked']}")
    except FileNotFoundError:
        print(label, "-> not run")

**Step 6b (only if you re-ran the training).** Compare with the first run. The checkpoint you hand over must come from the run whose numbers are reported.

In [ ]:
import json
first_run = {"resnet50_baseline_fmnist": (0.8346666666666667, 0.8330545225574444), "resnet50_finetune_fmnist": (0.8993333333333333, 0.9001889378071987)}
for run, (acc0, f10) in first_run.items():
    try:
        m = json.load(open(f"artifacts/m2/{run}/val_metrics.json"))
    except FileNotFoundError:
        print(run, "| not re-run in this session, skipped"); continue
    same = abs(m["accuracy"] - acc0) < 1e-9 and abs(m["macro_f1"] - f10) < 1e-9
    print(run, "| accuracy", round(m["accuracy"], 6), "vs first run", round(acc0, 6), "| macro-F1", round(m["macro_f1"], 6), "vs", round(f10, 6), "|", "IDENTICAL" if same else "DIFFERENT")

**Step 7.** Download the small result files (metrics, predictions, confusion matrix, class map). The checkpoint is NOT included; do not commit it.

In [ ]:
!cd artifacts/m2 && zip -qr ../../m2_results_small_files.zip . -i "*.json" "*.csv" -x "*.pt"
files.download("m2_results_small_files.zip")